# Chinook Database Analysis

SQL analysis of the [Chinook database](https://github.com/lerocha/chinook-database) — a sample SQLite database representing a small digital music store.

**Workflow:** Connect → Explore Schema → Query → Visualize

**Business questions answered:**
- Which are the 10 best-selling tracks?
- Which country generates the most revenue?
- Who is the top-performing sales employee?

## 1. Setup

Import required libraries and open a connection to the SQLite database file.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Open a connection to the Chinook SQLite database
conn = sqlite3.connect("Chinook_Sqlite.sqlite")

## 2. Schema Exploration

Retrieve all table names from the internal `sqlite_master` registry, then inspect the columns of the tables needed for the analysis.

In [ ]:
# List all tables in the database
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)
print(tables)

In [ ]:
# Inspect InvoiceLine and Track — needed for the best-selling tracks query
# Key columns: InvoiceLine.TrackId, InvoiceLine.Quantity | Track.TrackId, Track.Name
invoice_line = pd.read_sql("SELECT * FROM InvoiceLine LIMIT 5", conn)
track        = pd.read_sql("SELECT * FROM Track LIMIT 5", conn)

print("InvoiceLine:")
print(invoice_line)
print("\nTrack:")
print(track)

In [ ]:
# Inspect Customer and Invoice — needed for revenue by country and sales employee queries
# Key columns: Customer.SupportRepId | Invoice.BillingCountry, Invoice.Total, Invoice.CustomerId
customer = pd.read_sql("SELECT * FROM Customer LIMIT 5", conn)
invoice  = pd.read_sql("SELECT * FROM Invoice LIMIT 5", conn)

print("Customer:")
print(customer)
print("\nInvoice:")
print(invoice)

In [ ]:
# Inspect Employee — needed for the top-performing sales employee query
# Key columns: Employee.EmployeeId, Employee.FirstName, Employee.LastName
employee = pd.read_sql("SELECT * FROM Employee LIMIT 5", conn)
print(employee)

## 3. Business Questions

### 3.1 Top 10 Best-Selling Tracks

Join `Track` and `InvoiceLine` on `TrackId`, then sum `Quantity` sold per track.
Grouping by `TrackId` (primary key) rather than `Name` avoids merging distinct tracks that share the same title.

In [ ]:
# Top 10 best-selling tracks by total units sold
top_tracks = pd.read_sql(
    "SELECT Track.Name, SUM(InvoiceLine.Quantity) AS Units_Sold "
    "FROM Track "
    "JOIN InvoiceLine ON Track.TrackId = InvoiceLine.TrackId "
    "GROUP BY Track.TrackId "
    "ORDER BY Units_Sold DESC, Track.TrackId ASC "
    "LIMIT 10",
    conn
)
print(top_tracks)

### 3.2 Country with Highest Revenue

`Invoice` contains both `BillingCountry` and the pre-computed `Total` per invoice — no join required.
Sum the totals by country and return the top result.

In [ ]:
# Country that generates the most revenue
top_country = pd.read_sql("""
    SELECT BillingCountry, SUM(Total) AS Total_Revenue
    FROM Invoice
    GROUP BY BillingCountry
    ORDER BY Total_Revenue DESC
    LIMIT 1
""", conn)
print(top_country)

### 3.3 Top-Performing Sales Employee

Each customer is assigned to a sales support agent via `Customer.SupportRepId → Employee.EmployeeId`.
Chaining this with `Invoice` on `CustomerId` attributes each invoice's revenue to the responsible employee.

In [ ]:
# All sales employees ranked by total revenue from their assigned customers
sales_performance = pd.read_sql("""
    SELECT Employee.FirstName, Employee.LastName, SUM(Invoice.Total) AS Total_Revenue
    FROM Employee
    JOIN Customer ON Employee.EmployeeId = Customer.SupportRepId
    JOIN Invoice   ON Customer.CustomerId  = Invoice.CustomerId
    GROUP BY Employee.EmployeeId
    ORDER BY Total_Revenue DESC
""", conn)
print(sales_performance)

## 4. Visualization

Bar chart of the top 10 countries by revenue.
The best-selling tracks are not charted here because all tie at 2 units sold in this dataset — flat bars would carry no analytical value.

In [ ]:
# Fetch top 10 countries by revenue
chart_data = pd.read_sql("""
    SELECT BillingCountry, SUM(Total) AS Total_Revenue
    FROM Invoice
    GROUP BY BillingCountry
    ORDER BY Total_Revenue DESC
    LIMIT 10
""", conn)

# Bar chart: revenue by country
plt.bar(chart_data["BillingCountry"], chart_data["Total_Revenue"])
plt.title("Revenue by Country — Top 10")
plt.xlabel("Country")
plt.ylabel("Revenue (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Close Connection

Release the database file handle when the analysis is complete.

In [ ]:
conn.close()
print("Connection closed.")